In [5]:
# %%
import pickle
import pandas as pd
from IPython.display import display, Markdown

# ==========================================
# ⏱️ 量化时光机：机构级每日战报 (升级版)
# ==========================================
TARGET_DATE = "2026-05-08"  # 你可以随时改成其他交易日
report_file = f"Daily_Reports/Report_{TARGET_DATE}.pkl"

try:
    with open(report_file, 'rb') as f:
        snap = pickle.load(f)

    # ✨ 核心修复：提取最强与次强因子，建立准确的内存变量映射
    king_f = snap.get('king_factor', 'Momentum')  # 增加容错，兜底默认主线
    sub_factor = snap.get('sub_factor', '未知')

    # 标题头
    display(Markdown(f"# 📊 宽客实战日报 | 交易日: {snap['date']}"))
    display(Markdown(
        f"### 👑 市场绝对主线: **`{snap['king_factor']}`** | 🥈 次强辅线: **`{sub_factor}`**"))
    display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 1：OLS 市场风向归因
    # ---------------------------------------------------------
    r2_fac = snap['ols_report']['r2_factors'] * 100
    r2_ind = snap['ols_report']['r2_ind'] * 100
    display(Markdown(
        f"#### 📡 1. 市场微观结构探测 (解释力 - 因子: `{r2_fac:.2f}%` | 行业: `{r2_ind:.2f}%`)"))

    style_ols = snap['ols_report']['stats_df'].style.map(
        lambda x: 'color: red; font-weight: bold' if isinstance(x, str) and '追捧' in x else
                  ('color: green; font-weight: bold' if isinstance(x, str)
                   and '抛售' in x else ''),
        subset=['风向状态']
    ).background_gradient(subset=['Beta系数'], cmap='coolwarm', vmin=-0.05, vmax=0.05)
    display(style_ols)

    display(Markdown(f"#### 🏆 1.5 因子全维度强度对比"))
    king_stats_styled = snap['king_stats'].style.background_gradient(
        subset=['Beta (强度)'], cmap='RdYlGn'
    ).map(
        lambda x: 'font-weight: bold; color: #FF4500' if isinstance(
            x, (int, float)) and abs(x) > 1.96 else '',
        subset=['T值 (显著性)']
    )
    display(king_stats_styled)
    
    # ---------------------------------------------------------
    # 模块 2：【全新】板块时序动量与资金跃迁监控
    # ---------------------------------------------------------
    if 'sector_rank' in snap:
        display(Markdown("#### 🔄 2. 核心主舞台轮动透视 (已过滤低成交量噪音)"))
        rank_df = snap['sector_rank']

        # 提取排名飙升（抢筹）和暴跌（撤离）的板块
        soaring = rank_df.sort_values(by='Change', ascending=False).head(5)
        diving = rank_df.sort_values(by='Change', ascending=True).head(5)

        # ✨ 修复逻辑：格式化资金量（单位：亿元），让资金关注度一目了然
        for df in [soaring, diving]:
            if 'T0_Amount' in df.columns:
                df['成交额(亿)'] = (df['T0_Amount'] /
                                1e8).apply(lambda x: f"{x:.1f}")

        soaring_show = soaring[['T0_Rank', 'Change', '成交额(亿)']].rename(
            columns={'T0_Rank': '当前排名', 'Change': '跃升(位)'}
        )
        diving_show = diving[['T0_Rank', 'Change', '成交额(亿)']].rename(
            columns={'T0_Rank': '当前排名', 'Change': '下跌(位)'}
        )

        # 美化并排展示
        display(pd.concat([soaring_show, diving_show],
                axis=1, keys=['🚀 主力抢筹飙升', '📉 主力退潮撤离']))
        display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 3：截面强势板块下钻与龙头画像
    # ---------------------------------------------------------
    display(Markdown("#### 🎯 3. 截面强势板块下钻与龙头画像"))
    col1, col2 = snap['drill_report']['top_industries'], snap['drill_report']['bottom_industries']
    display(pd.concat([col1.set_index('板块名称'), col2.set_index(
        '板块名称')], axis=1, keys=['🔥 领涨板块', '❄️ 领跌板块']))

    display(Markdown("**【强势板块内部资金画像】**"))
    display(snap['drill_report']['portraits_df'].style.set_properties(
        **{'text-align': 'left'}))
    display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 4：【修复重点】全市场(ZZ800) 终极猎物池
    # ---------------------------------------------------------
    if 'zz800_top100' in snap:
        display(Markdown(
            f"#### 🌐 4. 全市场(ZZ800) 复合多因子 Top 20 (70% `{king_f}` + 30% `{sub_factor}`)"))

        df_zz800 = snap['zz800_top100'].copy()

        # ✨ 修复逻辑：显式定义要展示的列，确保 'name' 在前排
        # 如果 'name' 缺失（显示为 NaN），则填充为 '未知'
        if 'name' in df_zz800.columns:
            df_zz800['name'] = df_zz800['name'].fillna('未匹配名称')
        else:
            df_zz800['name'] = '缺少名称列'

        # 定义展示列清单
        show_cols = ['code', 'name', 'Industry',
                     'Composite_Score', king_f, sub_factor]
        # 过滤掉不存在于 DataFrame 中的列，防止报错
        actual_show_cols = [c for c in show_cols if c in df_zz800.columns]

        # 渲染前 20 名
        top_20_view = df_zz800.head(20)[actual_show_cols]

        display(top_20_view.style.background_gradient(
            subset=['Composite_Score'], cmap='YlOrRd'
        ).format({
            'Composite_Score': "{:.2f}",
            king_f: "{:.2f}" if king_f in top_20_view.columns else "{}",
            sub_factor: "{:.2f}" if sub_factor in top_20_view.columns else "{}"
        }))
        display(Markdown("---"))

    # ---------------------------------------------------------
    # 模块 5：个人底仓洗牌监控
    # ---------------------------------------------------------
    display(
        Markdown(f"#### 🔫 5. 个人底仓洗牌池 (按主线 **`{snap['king_factor']}`** 降序排列)"))

    # 动态组装你想看的因子列，确保不重复
    display_cols = ['code', 'name', snap['king_factor'],
                    sub_factor, 'Momentum', 'Liquidity', 'Size']
    display_cols = list(dict.fromkeys(display_cols))  # 去重，保持顺序

    # 只取存在的列进行展示
    actual_cols = [c for c in display_cols if c in snap['top_picks'].columns]
    top_picks = snap['top_picks'].head(20)[actual_cols]

    display(top_picks.style.background_gradient(
        subset=[snap['king_factor']], cmap='YlOrRd'))


except FileNotFoundError:
    print(f"❌ 找不到 {TARGET_DATE} 的战报！可能当天未生成或日期错误。")
# %%

# 📊 宽客实战日报 | 交易日: 2026-05-08

### 👑 市场绝对主线: **`Momentum`** | 🥈 次强辅线: **`Short_Rev`**

---

#### 📡 1. 市场微观结构探测 (解释力 - 因子: `21.40%` | 行业: `27.31%`)

,因子 (Factor),Beta系数,T值 (显著性),风向状态
0,Momentum,22.370000,3.480000,追捧 ★
1,Short_Rev,29.511000,4.560000,追捧 ★
2,Low_Vol,-30.540000,-4.640000,抛售 ★
3,Liquidity,41.253000,6.590000,追捧 ★
4,Size,-16.403000,-2.520000,抛售 ★
5,Value_BP,-35.643000,-4.000000,抛售 ★
6,Mom_Sharpe,-1.262000,-0.190000,抛售
7,Vol_Price_Corr,-3.537000,-0.560000,抛售
8,Amihud,-7.730000,-1.160000,抛售


...

#### 🌐 4. 全市场(ZZ800) 复合多因子 Top 20 (70% `Momentum` + 30% `Short_Rev`)

,code,name,Industry,Composite_Score,Momentum,Short_Rev
0,sh.600707,彩虹股份,C39计算机、通信和其他电子设备制造业,2.29,3.22,0.12
1,sz.300604,长川科技,C35专用设备制造业,2.17,2.99,0.28
2,sh.688702,盛科通信,I65软件和信息技术服务业,2.06,3.22,-0.65
3,sz.002938,鹏鼎控股,C39计算机、通信和其他电子设备制造业,1.96,2.67,0.32
4,sz.001389,广合科技,C39计算机、通信和其他电子设备制造业,1.95,2.88,-0.22
5,sz.300408,三环集团,C39计算机、通信和其他电子设备制造业,1.95,2.94,-0.35
6,sz.002916,深南电路,C39计算机、通信和其他电子设备制造业,1.94,2.67,0.24
7,sh.688002,睿创微纳,C39计算机、通信和其他电子设备制造业,1.93,2.60,0.37
8,sz.300390,天华新能,C39计算机、通信和其他电子设备制造业,1.89,3.22,-1.19
9,sz.002475,立讯精密,C39计算机、通信和其他电子设备制造业,1.88,2.60,0.20


---

#### 🔫 5. 个人底仓洗牌池 (按主线 **`Momentum`** 降序排列)

,code,name,Momentum,Short_Rev,Liquidity,Size
0,688008,澜起科技,3.216896,-2.680140,1.614067,1.590129
1,688041,海光信息,2.689774,-1.285276,-0.264363,2.974407
2,688099,晶晨股份,2.668251,-2.113866,1.082448,-0.209518
3,688002,睿创微纳,2.602495,0.368258,0.084783,0.285830
4,300308,中际旭创,2.310667,-0.611913,0.312140,3.108967
5,688347,华虹公司,2.279541,-1.776633,1.885767,0.163174
6,688017,绿的谐波,2.218508,-3.085377,0.832751,-0.212505
7,338,潍柴动力,2.001078,-0.990668,0.246772,1.262458
8,2466,天齐锂业,1.708713,-0.228969,2.239169,0.881267
9,688375,国博电子,1.519962,-0.676845,-0.473779,0.502315
